<a href="https://colab.research.google.com/github/VictorJorge5/panel-vuelos/blob/main/Lambda_Ingest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:



import json
import boto3
import requests
from datetime import datetime, timezone

# --- PARCHE ROBUSTO PARA BROTLI ---
class FakeBrotli:
    class error(Exception): pass
    # Necesitamos que tenga al menos un método que devuelva el mismo contenido
    def decompress(self, data): return data
    # Este es el método que falta y causa el error:
    class Decompressor:
        def __init__(self): pass
        def decompress(self, data): return data
    # Mapeamos el acceso para que no falle
    def Decompressor(self): return self.Decompressor()

import sys
sys.modules['brotli'] = FakeBrotli()

from FlightRadar24 import FlightRadar24API

# --- CONFIGURACIÓN DE AEROPUERTOS ---
AEROPUERTOS = {
    "ATL": {"coords": [33.6407, -84.4277]},
    "ORD": {"coords": [41.9742, -87.9073]},
    "LAX": {"coords": [33.9416, -118.4085]},
    "JFK": {"coords": [40.6413, -73.7781]}
}

def lambda_handler(event, context):
    fr_api = FlightRadar24API()
    s3 = boto3.client('s3')
    BUCKET_NAME = 'tfm-vuelos-meteo-ml-grupo5-393229434118-eu-west-1-an'
    target_iatas = set(AEROPUERTOS.keys())

    ahora = datetime.now(timezone.utc)
    timestamp_str = ahora.strftime('%Y%m%d_%H%M%S')
    nombre_archivo = f"raw/snapshot_completo_{timestamp_str}.json"

    datos_finales = {
        "metadata": {
            "snapshot_id": f"snapshot_{timestamp_str}",
            "generado_en_utc": ahora.isoformat()
        },
        "vuelos_en_aire": [],
        "llegadas_programadas": [],
        "salidas_programadas": [],
        "meteo_detallada": {},
        "metar_taf": {}
    }

    # 1. TELEMETRÍA DE VUELOS EN AIRE (Radar) + Captura de ETA
    try:
        vuelos = fr_api.get_flights()
        for v in vuelos:
            dest = str(getattr(v, 'destination_airport_iata', 'N/A')).upper()
            orig = str(getattr(v, 'origin_airport_iata', 'N/A')).upper()

            if dest in target_iatas or orig in target_iatas:
                ref_apt = dest if dest in target_iatas else orig

                # Intentamos obtener información de tiempo para el radar (si está disponible)
                # Esto ayuda a la IA a saber cuándo aterrizará este avión que ya está volando
                datos_finales["vuelos_en_aire"].append({
                    "aeropuerto_referencia": ref_apt,
                    "callsign": v.callsign,
                    "origen": orig,
                    "destino": dest,
                    "latitud": v.latitude,
                    "longitud": v.longitude,
                    "altitud": v.altitude,
                    "rumbo": v.heading,
                    "velocidad_nudos": v.ground_speed,
                    "velocidad_vertical": getattr(v, 'vertical_speed', 0),
                    "matricula": getattr(v, 'registration', 'N/A'),
                    "modelo_avion": getattr(v, 'aircraft_code', 'N/A'),
                    "aerolinea_iata": getattr(v, 'airline_iata', 'N/A'),
                    "time_details": getattr(v, 'time', {}) # Añadimos nodo de tiempo
                })
    except Exception as e: print(f"Error Radar: {e}")

    # 2. PANELES DE LLEGADAS Y SALIDAS (Schedules con horarios fijos)
    for apt in target_iatas:
        try:
            detalles = fr_api.get_airport_details(apt)
            if 'pluginData' in detalles.get('airport', {}):
                sched = detalles['airport']['pluginData']['schedule']
                # Aquí 'f' ya es un diccionario complejo que contiene f['flight']['time']['scheduled']['arrival']
                for f in sched['arrivals']['data']:
                    f['target_apt'] = apt
                    datos_finales["llegadas_programadas"].append(f)
                for f in sched['departures']['data']:
                    f['target_apt'] = apt
                    datos_finales["salidas_programadas"].append(f)
        except Exception as e: print(f"Error Panel {apt}: {e}")

    # 3. METEOROLOGÍA COMPLETA (Previsiones por horas)
    url_meteo = "https://api.open-meteo.com/v1/forecast"
    for apt, info in AEROPUERTOS.items():
        params = {
            "latitude": info["coords"][0], "longitude": info["coords"][1],
            "hourly": "wind_speed_10m,wind_gusts_10m,wind_direction_10m,visibility,cloudcover,temperature_2m,precipitation",
            "wind_speed_unit": "kn", "precipitation_unit": "mm", "timezone": "UTC"
        }
        try:
            r = requests.get(url_meteo, params=params)
            if r.status_code == 200:
                h = r.json().get("hourly", {})
                tiempos = h.get("time", [])
                clima_hora = {}
                for i, t in enumerate(tiempos):
                    clima_hora[t] = {
                        'viento_kts': h["wind_speed_10m"][i],
                        'rafagas_kts': h["wind_gusts_10m"][i],
                        'direccion': h["wind_direction_10m"][i],
                        'visib_m': h["visibility"][i],
                        'nubes_pct': h["cloudcover"][i],
                        'temp_c': h["temperature_2m"][i],
                        'precip': h["precipitation"][i]
                    }
                datos_finales["meteo_detallada"][apt] = clima_hora
        except Exception as e: print(f"Error Meteo {apt}: {e}")

    # 4. REPORTES METAR / TAF (Texto oficial)
    for apt in target_iatas:
        icao = f"K{apt}"
        try:
            m = requests.get(f"https://aviationweather.gov/api/data/metar?ids={icao}&format=raw", timeout=5).text
            t = requests.get(f"https://aviationweather.gov/api/data/taf?ids={icao}&format=raw", timeout=5).text
            datos_finales["metar_taf"][apt] = {"metar": m.strip(), "taf": t.strip()}
        except: pass

    # 5. GUARDADO ÚNICO EN S3
    s3.put_object(
        Bucket=BUCKET_NAME,
        Key=nombre_archivo,
        Body=json.dumps(datos_finales, indent=4)
    )

    return {
        'statusCode': 200,
        'body': f'Archivo histórico guardado con horarios: {nombre_archivo}'
    }


